In [ ]:
import os, sys, json, joblib
import numpy as np
import pandas as pd

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", "..", ".."))
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

MODEL_PATH = os.path.join(REPO_ROOT,"src", "app", "train","models", "modeloptuna.pkl")  # your saved pipeline
THRESH_JOBLIB = os.path.join(REPO_ROOT, "models", "threshold.joblib")  # optional
OUT_PATH = os.path.join(REPO_ROOT, "data", "predictions", "predictions_repurchase.csv")
os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)

print("Repo root:", REPO_ROOT)
print("Model path:", MODEL_PATH)
print("Output path:", OUT_PATH)



Repo root: c:\Users\gabri\OneDrive\ProyectoFinalMLOps
Model path: c:\Users\gabri\OneDrive\ProyectoFinalMLOps\src\app\train\models\modeloptuna.pkl
Output path: c:\Users\gabri\OneDrive\ProyectoFinalMLOps\data\predictions\predictions_repurchase.csv


In [ ]:
from src.app.train.etl import UserGenerator
from src.app.train.feature_engineer import FeatureEngineer


In [ ]:
ug = UserGenerator()
ug.run_etl()              
df_raw = ug.df.copy()

# Feature engineering (builds Revenue, timelines, label y_repurchase_30d, etc.)
fe = FeatureEngineer(df_raw)
df_features = fe.run()

assert pd.api.types.is_datetime64_any_dtype(df_features["InvoiceDate"])
display(df_features.head())
print(df_features.dtypes.head(12))


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\src\app\train\feature_engineer.py:52: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(self.historial_compra)   # <-- ahora acepta g


,Description,InvoiceDate,Country,Quantity,Revenue,UnitPrice,CustomerID,n_past_invoices,prev_date,recency_days,spend_prior,qty_prior,avg_ticket_prior,avg_qty_per_invoice_prior,next_date,days_to_next,y_repurchase_30d
0,3D DOG PICTURE PLAYING CARDS,2010-07-12 14:57:00,Iceland,24,70.8,2.95,12347,0,NaT,9999,0.0,0,0.000000,0.000000,2010-07-12 14:57:00,0.0,1
1,AIRLINE BAG VINTAGE JET SET BROWN,2010-07-12 14:57:00,Iceland,4,17.0,4.25,12347,1,2010-07-12 14:57:00,0,70.8,24,70.800000,24.000000,2010-07-12 14:57:00,0.0,1
2,ALARM CLOCK BAKELIKE CHOCOLATE,2010-07-12 14:57:00,Iceland,4,15.0,3.75,12347,2,2010-07-12 14:57:00,0,87.8,28,43.900000,14.000000,2010-07-12 14:57:00,0.0,1
3,ALARM CLOCK BAKELIKE GREEN,2010-07-12 14:57:00,Iceland,4,15.0,3.75,12347,3,2010-07-12 14:57:00,0,102.8,32,34.266667,10.666667,2010-07-12 14:57:00,0.0,1
4,ALARM CLOCK BAKELIKE ORANGE,2010-07-12 14:57:00,Iceland,4,15.0,3.75,12347,4,2010-07-12 14:57:00,0,117.8,36,29.450000,9.000000,2010-07-12 14:57:00,0.0,1


Description        string[python]
InvoiceDate        datetime64[ns]
Country                    object
Quantity                    int64
Revenue                   float64
UnitPrice                 float64
CustomerID                  int64
n_past_invoices             int64
prev_date          datetime64[ns]
recency_days                int64
spend_prior               float64
qty_prior                   int64
dtype: object


In [ ]:
# Load pipeline
from joblib import load

pipe = load(MODEL_PATH)

pre = pipe.named_steps.get("preprocessor") or pipe.named_steps.get("pre")
if pre is None:
    raise RuntimeError("Couldn't find the preprocessor step in the pipeline (expected 'preprocessor' or 'pre').")

# Extract the input columns the preprocessor expects
transformer_map = {name: cols for name, trans, cols in pre.transformers_}

num_cols = list(transformer_map.get("num", []))
cat_cols = list(transformer_map.get("cat", []))
feat_cols = num_cols + cat_cols

missing = [c for c in feat_cols if c not in df_features.columns]
if missing:
    print("Missing columns in df_features:", missing)
else:
    print("OK — all expected columns present.")

X_new = df_features.reindex(columns=feat_cols)
X_new.shape


OK — all expected columns present.


(164094, 10)

In [ ]:
# Definition of the threshold
from src.app.train.train_mlflow_advance import TrainOptuna
from sklearn.metrics import precision_recall_curve, f1_score

target_column = "y_repurchase_30d"  

trainer = TrainOptuna(
    df=df_features,
    numeric_features=num_cols,
    categorical_features=cat_cols,
    target_column=target_column,
    n_trials=1,                      
    optimization_metric="roc_auc",   
)

X_train, X_test, y_train, y_test = trainer.train_test_split_by_quantiles()

# Probabilities on the test split using the trained pipeline
proba_test = pipe.predict_proba(X_test)[:, 1]

# Threshold that maximizes F1 on test
prec, rec, thr = precision_recall_curve(y_test.astype(int), proba_test)
if len(thr) > 0:
    f1s = [f1_score(y_test, (proba_test >= t).astype(int)) for t in thr]
    t_star = float(thr[int(np.argmax(f1s))])
else:
    t_star = 0.5  

print("Chosen decision_threshold (F1-optimal on test):", t_star)


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Rango total: 2010-01-12 08:26:00 → 2011-12-10 17:19:00 | cutoff: 2011-11-10 17:19:00
train_end: 2011-09-01 00:00:00
train: 116228 | test: 33708
pos_rate train=0.966 | test=0.966
Chosen decision_threshold (F1-optimal on test): 1.2678166674417695e-13


In [ ]:
# Predict on current batch and save
proba = pipe.predict_proba(X_new)[:, 1]
yhat  = (proba >= t_star).astype(int)

pred = df_features.copy()
pred["p_repurchase_30d"] = proba
pred["repurchase_flag"]  = yhat

pred.to_csv(OUT_PATH, index=False)
print("✅ predictions_repurchase.csv written at:", OUT_PATH)
display(pred[["CustomerID", "InvoiceDate", "p_repurchase_30d", "repurchase_flag"]].head(10))


✅ predictions_repurchase.csv written at: c:\Users\gabri\OneDrive\ProyectoFinalMLOps\data\predictions\predictions_repurchase.csv


,CustomerID,InvoiceDate,p_repurchase_30d,repurchase_flag
0,12347,2010-07-12 14:57:00,0.000084,1
1,12347,2010-07-12 14:57:00,0.285979,1
2,12347,2010-07-12 14:57:00,0.313387,1
3,12347,2010-07-12 14:57:00,0.324296,1
4,12347,2010-07-12 14:57:00,0.330431,1
5,12347,2010-07-12 14:57:00,0.334629,1
6,12347,2010-07-12 14:57:00,0.337851,1
7,12347,2010-07-12 14:57:00,0.340236,1
8,12347,2010-07-12 14:57:00,0.343515,1
9,12347,2010-07-12 14:57:00,0.349492,1
